# Fabric Architecture Review - Setup

Copyright (c) Microsoft Corporation. Licensed under the MIT License.

Run this notebook to create or update FAR in the current Fabric workspace:

- A **Lakehouse** and **Collect -> Analyze -> Report -> Gold** pipeline.
- Standalone **05_Agent** and **06_TargetedReviewSetup** notebooks for optional chat and FUAM-targeted reviews.
- A **governance model and report** when `DEPLOY_GOLD_REPORT="true"` (default).
- A separate **Workspace Owner model/report**, **07_OwnerAccessSync** and **08_OwnerAgent** when `DEPLOY_WORKSPACE_OWNER_REPORT="true"` (default: false).
- An optional **Fabric IQ estate ontology** when governance deployment and `DEPLOY_ONTOLOGY` are enabled (preview; requires its tenant setting).

## Before running

1. Use a Fabric/Premium-capacity workspace and an identity permitted to create its items. Configure collection access using the [authentication guide](https://github.com/microsoft/fabric-architecture-review/blob/main/docs/auth-setup.md).
2. Pause schedules and finish active runs before redeploying. Setup updates generated items and can overwrite manual edits. Recognized owner-model upgrades run in place; unrecognized contract changes or a genuinely different report binding stop deployment for review.
3. After setup, configure model connections and run a scoped review before enabling optional features. Setup does not run the pipeline, publish Agents, grant consumer access or enable schedules.

Owner reporting also requires a fixed-identity connection with SSO disabled, access synchronization, manual reader approval and live access tests. Follow the owner checklist below before sharing.

## Parameters

In [ ]:
# Parameters
# --- GitHub repo to clone ---
GITHUB_REPO_URL = "https://github.com/microsoft/fabric-architecture-review.git"
GITHUB_BRANCH = "main"
# Release pinning: blank = the tag matching VERSION (e.g. v2026.09.2); set to pin a release.
RELEASE_TAG = ""

# --- Deploy targets (where the items land) ---
WORKSPACE_ID = ""                 # blank = this notebook's workspace
LAKEHOUSE_NAME = "fabric_arch_review_lh"
PIPELINE_NAME = "Fabric Arch Review Pipeline"
NOTEBOOK_PREFIX = "FabricArchReview"

# --- Governance report (Direct Lake) deployed alongside the pipeline ---
SEMANTIC_MODEL_NAME = "Fabric Arch Review - Governance"
REPORT_NAME = "Fabric Arch Review - Governance"
DATA_AGENT_NAME = "Fabric Arch Review - Data Agent"
DEPLOY_GOLD_REPORT = "true"   # "false" to skip the model + report
DEPLOY_WORKSPACE_OWNER_REPORT = "false"  # opt-in; no automatic sharing or schedule
OWNER_SEMANTIC_MODEL_NAME = "Fabric Arch Review - Workspace Owner Model"
OWNER_REPORT_NAME = "Fabric Arch Review - Workspace Owner"
OWNER_AGENT_NAME = "Fabric Arch Review - Workspace Owner Agent"
# The data agent is deployed by the separate "05_Agent" notebook (run after the
# pipeline); it needs the fabric-data-agent-sdk which requires a %pip + kernel restart.
ENDORSE_MODEL = ""                # optional: "Promoted" or "Certified" (blank = skip)
SENSITIVITY_LABEL_ID = ""         # optional: sensitivity label GUID to apply (blank = skip)
ONTOLOGY_NAME = "Fabric_Arch_Review_Estate_Ontology"
DEPLOY_ONTOLOGY = "true"   # "false" to skip the Fabric IQ estate ontology

# --- Optional service principal (unattended / scheduled runs) ---
# SP_CLIENT_ID alone does not switch identity. Set SP_SECRET_KEYVAULT and
# SP_SECRET_NAME on the pipeline for the optional Key Vault credential path.
SP_CLIENT_ID = ""                 # SP app (client) id; blank = notebook identity
SP_CONNECTION_NAME = "sp-fabric-arch-review"  # compatibility setting; not used for authentication

# Everything else — client/engagement/reviewer names, tenant + workspace scope,
# the capacity-metrics flag, VertiPaq depth, and all analysis thresholds — is
# exposed as a *pipeline parameter*: set it per-run in the Fabric Run dialog. The
# baked-in defaults live in the "Deploy ... pipeline" cell below.

## 1. Clone the repo (to read the stage-notebook sources)

In [ ]:
if str(DEPLOY_WORKSPACE_OWNER_REPORT).strip().lower() not in ("true", "false"):
    raise ValueError("DEPLOY_WORKSPACE_OWNER_REPORT must be true or false.")
if str(DEPLOY_WORKSPACE_OWNER_REPORT).strip().lower() == "true":
    if not isinstance(OWNER_AGENT_NAME, str) or not OWNER_AGENT_NAME.strip():
        raise ValueError("OWNER_AGENT_NAME must be a nonempty string.")

import os, sys, shutil, subprocess
WORK_ROOT = "/tmp/fabric-arch-review-setup"
REPO_DIR = os.path.join(WORK_ROOT, "repo")
os.makedirs(WORK_ROOT, exist_ok=True)
_url = GITHUB_REPO_URL

def _fresh_clone():
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, "--depth", "1", _url, REPO_DIR], check=True)

# An existing single-branch clone won't have an origin/<branch> ref for a *different*
# branch, so reset to FETCH_HEAD (the tip we just fetched) instead. Refresh the remote
# URL first in case the repo changed; fall back to a clean clone on any failure.
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    try:
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", _url], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", GITHUB_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"], check=True)
    except subprocess.CalledProcessError:
        _fresh_clone()
else:
    _fresh_clone()
try:
    with open(os.path.join(REPO_DIR, "VERSION"), encoding="utf-8-sig") as _vf:
        _branch_version = _vf.read().strip()
except Exception:
    _branch_version = ""
RELEASE_TAG = (RELEASE_TAG or "").strip()
GITHUB_REF = RELEASE_TAG or (("v" + _branch_version) if _branch_version else GITHUB_BRANCH)
def _checkout_ref(ref):
    try:
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", ref], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", "--force", "FETCH_HEAD"], check=True)
        return True
    except subprocess.CalledProcessError:
        return False
if GITHUB_REF != GITHUB_BRANCH and _checkout_ref(GITHUB_REF):
    print("Deploying pinned release ref:", GITHUB_REF)
else:
    if GITHUB_REF != GITHUB_BRANCH:
        print("WARNING: tag '" + GITHUB_REF + "' not found - deploying '" + GITHUB_BRANCH + "' tip (unpinned).")
    GITHUB_REF = GITHUB_BRANCH
try:
    DEPLOY_SHA = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
except Exception:
    DEPLOY_SHA = ""
del _url
print("Repo cloned at", REPO_DIR, "| ref", GITHUB_REF, "| sha", DEPLOY_SHA[:8])

## 2. Fabric REST helpers

In [ ]:
import os, json, base64, time, requests, notebookutils, sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
for _m in [m for m in list(sys.modules) if m == 'orchestration' or m.startswith('orchestration.')]:
    del sys.modules[_m]
from orchestration.fabric_api import FabricClient
from orchestration.folders import find_item as _find_far_item
_folder_client = FabricClient(lambda: notebookutils.credentials.getToken('pbi'))
BASE = "https://api.fabric.microsoft.com/v1"

def _H():
    return {"Authorization": "Bearer " + notebookutils.credentials.getToken("pbi"),
            "Content-Type": "application/json"}

def _poll(resp):
    if resp.status_code in (200, 201):
        return resp.json() if resp.content else {}
    if resp.status_code == 202:
        loc = resp.headers.get("Location")
        while loc:
            r = requests.get(loc, headers=_H())
            j = r.json() if r.content else {}
            s = str(j.get("status", ""))
            if s in ("Succeeded", "Completed"):
                rr = requests.get(loc.rstrip("/") + "/result", headers=_H())
                return rr.json() if (rr.status_code == 200 and rr.content) else j
            if s in ("Failed", "Cancelled", "Canceled"):
                raise RuntimeError("LRO " + s + ": " + json.dumps(j))
            time.sleep(int(r.headers.get("Retry-After", "3")))
    if resp.status_code >= 400:
        raise RuntimeError("Fabric HTTP " + str(resp.status_code) + " " + (resp.request.method or "") + " " + str(resp.url) + " -> " + (resp.text or "")[:4000])
    resp.raise_for_status()
    return {}

def find_item(wid, name, itype):
    return _find_far_item(_folder_client, wid, name, itype)

def ensure_lakehouse(wid, name):
    it = find_item(wid, name, "Lakehouse")
    if it:
        return it["id"]
    res = _poll(requests.post(BASE + "/workspaces/" + wid + "/lakehouses", headers=_H(),
                              json={"displayName": name}))
    if res.get("id"):
        return res["id"]
    return find_item(wid, name, "Lakehouse")["id"]

def _upsert(wid, name, itype_path, part_path, fmt, content_obj):
    payload = base64.b64encode(json.dumps(content_obj).encode("utf-8")).decode("ascii")
    definition = {"parts": [{"path": part_path, "payload": payload, "payloadType": "InlineBase64"}]}
    if fmt:
        definition["format"] = fmt
    itype = "Notebook" if itype_path == "notebooks" else "DataPipeline"
    it = find_item(wid, name, itype)
    if it:
        _poll(requests.post(BASE + "/workspaces/" + wid + "/" + itype_path + "/" + it["id"] + "/updateDefinition",
                            headers=_H(), json={"definition": definition}))
        return it["id"]
    res = _poll(requests.post(BASE + "/workspaces/" + wid + "/" + itype_path, headers=_H(),
                              json={"displayName": name, "definition": definition}))
    if res.get("id"):
        return res["id"]
    return find_item(wid, name, itype)["id"]

def upsert_notebook(wid, name, nb):
    return _upsert(wid, name, "notebooks", "notebook-content.ipynb", "ipynb", nb)

def upsert_pipeline(wid, name, pl):
    return _upsert(wid, name, "dataPipelines", "pipeline-content.json", None, pl)

def get_json(path):
    r = requests.get(BASE + path, headers=_H())
    return r.json() if (r.status_code == 200 and r.content) else {}

def upsert_definition(wid, itype_path, itype, name, definition):
    it = find_item(wid, name, itype)
    if it:
        _poll(requests.post(BASE + "/workspaces/" + wid + "/" + itype_path + "/" + it["id"] + "/updateDefinition",
                            headers=_H(), json={"definition": definition}))
        return it["id"]
    res = _poll(requests.post(BASE + "/workspaces/" + wid + "/" + itype_path, headers=_H(),
                              json={"displayName": name, "definition": definition}))
    if res.get("id"):
        return res["id"]
    return find_item(wid, name, itype)["id"]

def ensure_sp_connection(name, client_id, tenant_id):
    # Idempotently create a shareable cloud connection holding a Service Principal.
    # Secret is intentionally blank: paste it once via Manage connections post-setup.
    r = requests.get(BASE + "/connections?$top=200", headers=_H())
    for c in (r.json().get("value", []) if r.status_code == 200 and r.content else []):
        if c.get("displayName") == name:
            return c["id"]
    body = {
        "connectivityType": "ShareableCloud", "displayName": name,
        "connectionDetails": {"type": "WebForPipeline", "creationMethod": "WebForPipeline.Contents",
                              "parameters": [{"dataType": "Text", "name": "url", "value": "https://api.fabric.microsoft.com"}]},
        "privacyLevel": "Organizational",
        "credentialDetails": {"singleSignOnType": "None", "connectionEncryption": "NotEncrypted",
                              "skipTestConnection": True,
                              "credentials": {"credentialType": "ServicePrincipal", "tenantId": tenant_id,
                                              "servicePrincipalClientId": client_id, "servicePrincipalSecret": ""}},
    }
    r = requests.post(BASE + "/connections", headers=_H(), json=body)
    if r.status_code in (200, 201) and r.content:
        return r.json().get("id")
    print("  connection create failed:", r.status_code, (r.text or "")[:500])
    return name + " (create manually)"

print("REST helpers ready.")

## 3. Deploy Lakehouse + stage notebooks + pipeline

In [ ]:
import json, os, notebookutils
ctx = notebookutils.runtime.context
wid = WORKSPACE_ID or ctx.get("currentWorkspaceId") or ctx.get("workspaceId")
print("Target workspace:", wid)

lhid = ensure_lakehouse(wid, LAKEHOUSE_NAME)
print("Lakehouse:", LAKEHOUSE_NAME, lhid)

def load_nb(rel):
    with open(os.path.join(REPO_DIR, rel), encoding="utf-8") as f:
        nb = json.load(f)
    dep = nb.setdefault("metadata", {}).setdefault("dependencies", {})
    dep["lakehouse"] = {
        "default_lakehouse": lhid,
        "default_lakehouse_name": LAKEHOUSE_NAME,
        "default_lakehouse_workspace_id": wid,
        "known_lakehouses": [{"id": lhid}],
    }
    return nb

def load_agent_nb():
    nb = load_nb("fabric/notebooks/05_agent.ipynb")
    values = {
        "GITHUB_REPO_URL": GITHUB_REPO_URL,
        "GITHUB_BRANCH": GITHUB_BRANCH,
        "GITHUB_REF": GITHUB_REF,
        "DATA_AGENT_NAME": DATA_AGENT_NAME,
        "SEMANTIC_MODEL_NAME": SEMANTIC_MODEL_NAME,
        "LAKEHOUSE_NAME": LAKEHOUSE_NAME,
    }
    replaced = set()
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = cell.get("source", [])
        for index, line in enumerate(source):
            stripped = line.lstrip()
            for name, value in values.items():
                if stripped.startswith(name + " ") and "=" in stripped:
                    line_body = line.rstrip("\r\n")
                    line_ending = line[len(line_body):]
                    comment = ""
                    if "#" in line_body:
                        comment = "  #" + line_body.split("#", 1)[1]
                    source[index] = name + " = " + json.dumps(str(value)) + comment + line_ending
                    replaced.add(name)
                    break
    missing = set(values) - replaced
    if missing:
        raise RuntimeError("Could not stamp agent notebook deployment ref: " + ", ".join(sorted(missing)))
    return nb

collect_id = upsert_notebook(wid, NOTEBOOK_PREFIX + "_01_Collect", load_nb("fabric/notebooks/01_collect.ipynb"))
analyze_id = upsert_notebook(wid, NOTEBOOK_PREFIX + "_02_Analyze", load_nb("fabric/notebooks/02_analyze.ipynb"))
report_id = upsert_notebook(wid, NOTEBOOK_PREFIX + "_03_Report", load_nb("fabric/notebooks/03_report.ipynb"))
gold_id = upsert_notebook(wid, NOTEBOOK_PREFIX + "_04_Gold", load_nb("fabric/notebooks/04_gold.ipynb"))
agent_id = upsert_notebook(wid, NOTEBOOK_PREFIX + "_05_Agent", load_agent_nb())
print("Notebooks:", collect_id, analyze_id, report_id, gold_id, agent_id)


# --- Engagement + analysis pipeline parameters (set per-run in the Fabric Run
# dialog). These are NOT setup variables; their baked-in defaults live here. The
# activities below read each one via @pipeline().parameters.* so every value is
# selectable and overridable per run / on scheduled triggers. ---
common_keys = {
    "GITHUB_REPO_URL": GITHUB_REPO_URL, "GITHUB_BRANCH": GITHUB_BRANCH, "GITHUB_REF": GITHUB_REF,
    "CLIENT_NAME": "Contoso",
    "ENGAGEMENT_NAME": "Fabric Architecture Review",
    "REVIEWER_NAME": "",
}
collect_keys = {
    "TENANT_ID": "",                      # blank = this notebook's home tenant
    "WORKSPACE_IDS": "",                  # restrict to specific workspace GUIDs (blank = tenant-wide)
    "CAPACITY_METRICS_APP_INSTALLED": "false",
    "VERTIPAQ_STATS_READ_DATA": "false",  # "true" adds exact column cardinality (aggregate COUNT DAX)
    "ACTIVITY_DAYS_LOG": "7",             # activity-log lookback window in days (1-28)
    # Optional standing/unattended service-principal. Blank = run as the notebook
    # identity (default). Set SP_CLIENT_ID, SP_SECRET_KEYVAULT and SP_SECRET_NAME
    # together, plus TENANT_ID, for the explicit Key Vault credential path.
    "SP_CLIENT_ID": SP_CLIENT_ID,         # SP appId; blank = notebook identity
    "SP_CONNECTION_NAME": SP_CONNECTION_NAME,  # compatibility setting, not credential wiring
    "SP_SECRET_KEYVAULT": "",             # optional Key Vault name/URI holding the SP secret
    "SP_SECRET_NAME": "",                 # optional secret name for the SP override
}
# Analysis thresholds (only consumed by the Analyze stage). Blank = use the curated
# default from config/thresholds.yaml; a value here overrides it. The comment after
# each key shows the built-in default.
threshold_keys = {
    "ARCH_MONOLITH_THRESHOLD": "",        # 50   max items before a workspace is "monolithic"
    "ARCH_PIPELINE_STALE_DAYS": "",       # 30   days without a pipeline run = stale
    "PERF_STALE_DAYS": "",                # 30   days without a model refresh = stale
    "PERF_LONG_REFRESH_HOURS": "",        # 2.0  refresh duration (h) flagged as long
    "PERF_FAIL_RATIO_THRESHOLD": "",      # 0.2  refresh failure ratio flagged
    "PERF_MODEL_SIZE_WARN_MB": "",        # 2048 model size (MB) warn
    "PERF_MODEL_SIZE_CRITICAL_MB": "",    # 8192 model size (MB) critical
    "PERF_REFRESH_OVERLAP_MIN": "",       # 2    overlap (min) between refreshes flagged
    "PERF_THROTTLE_CRITICAL_PCT": "",     # 100  throttle % critical
    "PERF_THROTTLE_WARN_PCT": "",         # 70   throttle % warn
    "PERF_JOB_FAIL_RATIO": "",            # 0.2  job failure ratio flagged
    "PERF_JOB_LONG_HOURS": "",            # 1.0  job duration (h) flagged as long
    "GOV_SHARE_VOLUME_THRESHOLD": "",     # 100  per-item shares before "broadly shared"
    "SEC_BROAD_ACCESS_THRESHOLD": "",     # 10   workspace principals = broad access
    "SEC_GATEWAY_MIN_VERSION": "",        # ""   minimum acceptable gateway version (blank = no check)
    "COST_SMALL_WORKSPACE_THRESHOLD": "", # 5    min workspaces to justify a large SKU
    "COST_CONSOLIDATION_MIN_SMALL_CAPS": "", # 3 small F1-F16 capacities before "consolidate" (COST-007)
    "OPS_PROD_PIPELINE_MIN_RATIO": "",    # 0.8  prod workspaces on a deployment pipeline (OPS-001)
    "OPS_PROD_GIT_MIN_RATIO": "",         # 0.8  prod workspaces under Git (OPS-002)
    "GOV_ENDORSEMENT_MIN_RATIO": "",      # 0.3  endorsed-item coverage advisory (GOV-008)
    "GOV_PROD_ENDORSEMENT_MIN_RATIO": "", # 0.5  certified prod semantic models (GOV-009)
}

# Expose every value as a pipeline-level parameter so it is selectable and
# overridable in the Fabric Run dialog and on scheduled triggers (the activities
# below read them via @pipeline().parameters.* - the same way RUN_ID resolves).
_pp = {**common_keys, **collect_keys, **threshold_keys}
pipeline_parameters = {k: {"type": "string", "defaultValue": str(v)} for k, v in _pp.items()}

common = {k: "@pipeline().parameters." + k for k in common_keys}
common["RUN_ID"] = "@pipeline().RunId"
collect_params = dict(common)
collect_params.update({k: "@pipeline().parameters." + k for k in collect_keys})
analyze_params = dict(common)
analyze_params.update({k: "@pipeline().parameters." + k for k in threshold_keys})

def _P(d):
    return {k: {"value": v, "type": "string"} for k, v in d.items()}

def activity(name, nbid, params, depends=None):
    return {
        "name": name, "type": "TridentNotebook", "dependsOn": depends or [],
        "policy": {"timeout": "0.12:00:00", "retry": 1, "retryIntervalInSeconds": 120,
                   "secureOutput": False, "secureInput": False},
        "typeProperties": {"notebookId": nbid, "workspaceId": wid, "parameters": _P(params)},
    }

pipe = {"properties": {"parameters": pipeline_parameters, "activities": [
    activity("01 Collect", collect_id, collect_params),
    activity("02 Analyze", analyze_id, analyze_params, [{"activity": "01 Collect", "dependencyConditions": ["Succeeded"]}]),
    activity("03 Report", report_id, common, [{"activity": "02 Analyze", "dependencyConditions": ["Succeeded"]}]),
    activity("04 Gold", gold_id, common, [{"activity": "03 Report", "dependencyConditions": ["Succeeded"]}]),
]}}

pid = upsert_pipeline(wid, PIPELINE_NAME, pipe)
print("Pipeline:", PIPELINE_NAME, pid)

# Optional feature configuration is deployed, but never enabled by base setup.
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
for _module in [key for key in sys.modules if key == "orchestration" or key.startswith("orchestration.")]:
    del sys.modules[_module]
from orchestration.deployment import stamp_parameters

# --- Optional: Service Principal credential configuration ---------------------
# An app ID alone does not activate the Collect stage's Key Vault override.
SP_CLIENT_ID = (collect_keys.get("SP_CLIENT_ID") or "").strip()
SP_CONNECTION_NAME = (collect_keys.get("SP_CONNECTION_NAME") or "sp-fabric-arch-review").strip()
if SP_CLIENT_ID:
    print("SP_CLIENT_ID is configured. To use the Collect stage's service-principal override, "
          "set TENANT_ID, SP_SECRET_KEYVAULT and SP_SECRET_NAME on the pipeline and grant the "
          "notebook identity access to that secret. SP_CONNECTION_NAME does not switch identity.")

print("\nDeployed. Open the workspace and run '" + PIPELINE_NAME + "' (or schedule it).")
print("Outputs land in Lakehouse '" + LAKEHOUSE_NAME + "' under Files/fabric-arch-review/<run-id>/report.md")

## 4. Deploy reporting artifacts (governance, optional owner report and ontology)

In [ ]:
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Drop cached copies so a re-run picks up freshly cloned code (avoids stale-import bugs)
for _m in [m for m in list(sys.modules) if m == 'reports' or m.startswith('reports.')]:
    del sys.modules[_m]

if str(DEPLOY_GOLD_REPORT).strip().lower() in ("1", "true", "yes", "y", "on"):
    from reports.powerbi.deploy import wait_for_sql_endpoint, model_definition, report_definition
    print("Resolving Lakehouse SQL endpoint (can take ~1 min on a fresh Lakehouse)...")
    sql_server, sql_db = wait_for_sql_endpoint(get_json, wid, lhid)
    print("  SQL endpoint:", sql_server, "| database:", sql_db)

    # Bootstrap empty gold Delta tables so the Direct Lake model + report deploy
    # (and validate column references) even before the pipeline has ever run.
    # mode("ignore") is idempotent: it never clobbers tables the pipeline already filled.
    from pyspark.sql import SparkSession
    from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                                   DoubleType, BooleanType, TimestampType)
    from reports.powerbi.schema import GOLD_TABLES
    _spark = SparkSession.builder.getOrCreate()
    _T = {"string": StringType(), "int64": LongType(), "double": DoubleType(),
          "boolean": BooleanType(), "dateTime": TimestampType()}
    _tbl_root = "abfss://" + wid + "@onelake.dfs.fabric.microsoft.com/" + lhid + "/Tables"
    print("Bootstrapping empty gold tables (skipped where the pipeline already wrote data)...")
    for _t in GOLD_TABLES:
        _sch = StructType([StructField(c.name, _T[c.kind], True) for c in _t.columns])
        _spark.createDataFrame([], _sch).write.format("delta").mode("ignore").save(_tbl_root + "/" + _t.name)
    print("  gold tables ready:", ", ".join(t.name for t in GOLD_TABLES))

    # Record the DEPLOYED FAR version (from the cloned VERSION file) into a small
    # meta_deployment Delta table. The 04_Gold stage reads it to build gold_release
    # and the report's version banner flags when a newer release exists. Overwrite
    # keeps a single current row; it survives pipeline runs and is refreshed only
    # when you re-run setup.ipynb (i.e. when you actually update). Lakehouse history
    # tables (gold_*) are never touched here.
    from datetime import datetime, timezone
    _far_ver = "unknown"
    try:
        with open(os.path.join(REPO_DIR, "VERSION"), encoding="utf-8-sig") as _vf:
            _far_ver = _vf.read().strip() or "unknown"
    except Exception as _e:
        print("  (could not read VERSION file:", _e, ")")
    _meta_sch = StructType([
        StructField("version", StringType(), True),
        StructField("deployed_at_utc", StringType(), True),
        StructField("repo_url", StringType(), True),
        StructField("branch", StringType(), True),
        StructField("git_ref", StringType(), True),
        StructField("git_sha", StringType(), True),
    ])
    _meta_row = [(_far_ver, datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
                  GITHUB_REPO_URL, GITHUB_BRANCH, GITHUB_REF, DEPLOY_SHA)]
    (_spark.createDataFrame(_meta_row, _meta_sch).write.format("delta")
        .mode("overwrite").option("overwriteSchema", "true").save(_tbl_root + "/meta_deployment"))
    print("  meta_deployment recorded: FAR v" + _far_ver + " @ " + GITHUB_REF)
    mid = upsert_definition(wid, "semanticModels", "SemanticModel", SEMANTIC_MODEL_NAME,
                            model_definition(SEMANTIC_MODEL_NAME, sql_server, sql_db))
    print("Semantic model:", SEMANTIC_MODEL_NAME, mid)
    rid = upsert_definition(wid, "reports", "Report", REPORT_NAME, report_definition(mid))
    print("Report:", REPORT_NAME, rid)
    if ENDORSE_MODEL or SENSITIVITY_LABEL_ID:
        try:
            _lbl_items = [mid, rid]
            if ENDORSE_MODEL:
                from sempy_labs.semantic_model import set_endorsement
                set_endorsement(dataset=SEMANTIC_MODEL_NAME, endorsement=ENDORSE_MODEL, workspace=wid)
                print("Endorsed model as", ENDORSE_MODEL)
            if SENSITIVITY_LABEL_ID:
                from sempy_labs.admin import bulk_set_labels
                bulk_set_labels(item_ids=[i for i in _lbl_items if i], label_id=SENSITIVITY_LABEL_ID)
                print("Applied sensitivity label to model/report")
        except Exception as _e:
            print("Endorsement/label skipped (best-effort; needs admin rights + valid label id):", str(_e)[:200])
    if DEPLOY_ONTOLOGY == "true":
        try:
            from reports.ontology.ontology import ontology_definition
            _ondef = ontology_definition(lhid, wid, name=ONTOLOGY_NAME)
            _on = find_item(wid, ONTOLOGY_NAME, "Ontology")
            if _on:
                _poll(requests.post(BASE + "/workspaces/" + wid + "/items/" + _on["id"] + "/updateDefinition",
                                    headers=_H(), json={"definition": _ondef}))
                oid = _on["id"]
            else:
                _res = _poll(requests.post(BASE + "/workspaces/" + wid + "/items", headers=_H(),
                                           json={"displayName": ONTOLOGY_NAME, "type": "Ontology", "definition": _ondef}))
                oid = _res.get("id") or (find_item(wid, ONTOLOGY_NAME, "Ontology") or {}).get("id")
            print("Ontology:", ONTOLOGY_NAME, oid)
        except Exception as _e:
            print("Ontology skipped (best-effort): enable the 'Users can create Ontology (preview) items' tenant setting, then re-run.", str(_e)[:200])
    print("\nGovernance report deployed. It shows data after the pipeline runs once")
    print("(deploy bootstraps the tables empty; the 04 Gold stage fills them each run).")
    print("Then run the '" + NOTEBOOK_PREFIX + "_05_Agent' notebook once to deploy + publish the data agent.")
else:
    print("DEPLOY_GOLD_REPORT is false -> skipped the governance model + report.")

owner_access_notebook_id = ""
owner_model_id = ""
if str(DEPLOY_WORKSPACE_OWNER_REPORT).strip().lower() not in ("true", "false"):
    raise ValueError("DEPLOY_WORKSPACE_OWNER_REPORT must be true or false.")
if str(DEPLOY_WORKSPACE_OWNER_REPORT).strip().lower() == "true":
    from pathlib import Path
    from pyspark.sql import SparkSession
    from orchestration.fabric_api import FabricClient
    from reports.owner.deployment import deploy_owner_reporting
    if (OWNER_SEMANTIC_MODEL_NAME == SEMANTIC_MODEL_NAME or OWNER_REPORT_NAME == REPORT_NAME):
        raise ValueError("Owner artifacts must have names distinct from governance artifacts.")
    owner_artifacts = deploy_owner_reporting(
        FabricClient(lambda: notebookutils.credentials.getToken("pbi")),
        repo_dir=Path(REPO_DIR), workspace_id=wid, lakehouse_id=lhid,
        lakehouse_name=LAKEHOUSE_NAME, notebook_prefix=NOTEBOOK_PREFIX,
        model_name=OWNER_SEMANTIC_MODEL_NAME, report_name=OWNER_REPORT_NAME,
        spark=SparkSession.builder.getOrCreate(),
    )
    owner_access_notebook_id = owner_artifacts["access_notebook_id"]
    owner_model_id = owner_artifacts["model_id"]

if str(DEPLOY_WORKSPACE_OWNER_REPORT).strip().lower() == "true":
    if not owner_model_id:
        raise ValueError("Owner Agent notebook requires a successfully deployed owner model.")
    _owner_agent_nb = stamp_parameters(load_nb("fabric/notebooks/08_owner_agent.ipynb"), {
        "GITHUB_REPO_URL": GITHUB_REPO_URL, "GITHUB_BRANCH": GITHUB_BRANCH, "GITHUB_REF": GITHUB_REF,
        "WORKSPACE_ID": wid, "OWNER_SEMANTIC_MODEL_ID": owner_model_id,
        "OWNER_AGENT_NAME": OWNER_AGENT_NAME.strip(),
    })
    owner_agent_notebook_id = upsert_notebook(wid, NOTEBOOK_PREFIX + "_08_OwnerAgent", _owner_agent_nb)
    print("Optional owner Agent notebook:", owner_agent_notebook_id)
    print("After Gold and successful 07 access sync, run 08 separately. Setup has not published or shared an owner Agent.")

# Bind 06 only after optional owner deployment has returned the existing 07 ID.
_targeted_nb = stamp_parameters(load_nb("fabric/notebooks/06_targeted_review_setup.ipynb"), {
    "GITHUB_REPO_URL": GITHUB_REPO_URL, "GITHUB_BRANCH": GITHUB_BRANCH, "GITHUB_REF": GITHUB_REF,
    "WORKSPACE_ID": wid, "LAKEHOUSE_ID": lhid, "CHILD_PIPELINE_ID": pid,
    "PARENT_PIPELINE_NAME": PIPELINE_NAME + " - Targeted Review",
    "OWNER_ACCESS_NOTEBOOK_ID": owner_access_notebook_id,
})
targeted_setup_id = upsert_notebook(wid, NOTEBOOK_PREFIX + "_06_TargetedReviewSetup", _targeted_nb)
print("Optional targeted-review setup notebook:", targeted_setup_id, "(disabled until explicitly configured)")


## Organize FAR artifacts

Reuse exact root folders and move existing item IDs in place, including previously deployed optional items. No items are deleted, shared, or scheduled. Folder API errors stop setup; resolve them and rerun.


In [ ]:
from orchestration.folders import organize, organize_existing
_known = [(lhid, 'Lakehouse', None), (pid, 'DataPipeline', 'Pipelines'),
          (targeted_setup_id, 'Notebook', 'Notebooks')]
_known += [(i, 'Notebook', 'Notebooks') for i in
           (collect_id, analyze_id, report_id, gold_id, agent_id)]
_configured = [
    (SEMANTIC_MODEL_NAME, 'SemanticModel', 'Reporting'),
    (REPORT_NAME, 'Report', 'Reporting'),
    (OWNER_SEMANTIC_MODEL_NAME, 'SemanticModel', 'Reporting'),
    (OWNER_REPORT_NAME, 'Report', 'Reporting'),
    (ONTOLOGY_NAME, 'Ontology', 'Ontology'),
    (DATA_AGENT_NAME, 'DataAgent', 'Agents'),
    (OWNER_AGENT_NAME, 'DataAgent', 'Agents'),
    (NOTEBOOK_PREFIX + '_07_OwnerAccessSync', 'Notebook', 'Notebooks'),
    (NOTEBOOK_PREFIX + '_08_OwnerAgent', 'Notebook', 'Notebooks'),
    (PIPELINE_NAME + ' - Targeted Review', 'DataPipeline', 'Pipelines'),
    (PIPELINE_NAME + ' - Targeted Review - Runner', 'Notebook', 'Notebooks'),
    (PIPELINE_NAME + ' - Targeted Review - Completion', 'Notebook', 'Notebooks'),
]
organize_existing(_folder_client, wid, _configured, known=_known)
_setup_workspace = ctx.get('currentWorkspaceId') or ctx.get('workspaceId')
_setup_id = ctx.get('currentNotebookId')
if not _setup_workspace or not _setup_id:
    raise ValueError('Runtime context must identify the imported setup notebook to keep it at root.')
organize(_folder_client, _setup_workspace, [(_setup_id, 'Notebook', None)])
print('FAR folders reconciled; item IDs, permissions and schedules preserved.')


## Optional Workspace Owner report - complete before sharing

Set `DEPLOY_WORKSPACE_OWNER_REPORT="true"` for curated, workspace-scoped reporting. Its RLS does not secure the central report, app or central Agent.

Complete these steps in order; use the [owner reporting guide](https://github.com/microsoft/fabric-architecture-review/blob/main/docs/workspace-owner-report.md) for the full walkthrough and acceptance checklist.

1. **Connect the owner semantic model.** In **Settings -> Gateway and cloud connections**, map it to a shared cloud fixed-identity connection for the FAR Lakehouse SQL endpoint. The connection identity needs source read access. Keep **Microsoft Entra SSO disabled**, apply the mapping and verify refresh. Reuse the connection on reruns; it is separate from Outlook sign-in.
2. **Populate owner evidence.** Run an owner-enabled, scoped FAR review through Gold. Existing governance-only history is not automatically projected into owner tables.
3. **Synchronize access.** Run **07_OwnerAccessSync** with the FAR Lakehouse attached, or wait for it in an owner-bound targeted parent. Rerun 06 after enabling owner reporting to update the parent. Sync covers all owner-projected workspaces, not only Top N, and does not grant report permissions.
4. **Approve a test reader.** Grant item-scoped **Read** on the owner report/model, with Build and resharing disabled. In the owner model's **Security -> WorkspaceOwner**, add the reader and save.
5. **Validate and schedule.** Test actual read-only users, denied workspaces, expiry, revocation and exports before broader sharing. Schedule 07 **daily**, monitor failures and warnings, and never overlap daily, targeted or manual syncs. Grants expire after **24 hours**. Use the validated owner report URL for notifications.

**Do not grant owner consumers a FAR workspace role, even Viewer, raw Lakehouse/SQL access, Build or Reshare.** Apply organizational labels and sharing restrictions separately; they do not configure RLS. Setup, sync and email never approve reader permissions automatically.

Setup reuses compatible models and upgrades recognized older contracts in place while preserving memberships and connections. Unrecognized changes stop deployment for review. Disabling the deployment flag does not revoke access, stop schedules or delete owner tables.

### Optional Workspace Owner Agent

After Gold and successful 07, run **08_OwnerAgent** separately using its stamped owner model ID. Follow its install/restart and deployment steps.

The secured owner model is its sole source. Approve **query-only** Agent access, model **Read** and `WorkspaceOwner` membership; Build is not required. Complete actual-reader access tests before sharing. Do not substitute the central Agent or app. See the [owner Agent guide](https://github.com/microsoft/fabric-architecture-review/blob/main/docs/workspace-owner-agent.md).

## 5. Deploy the central data agent (after the pipeline)

1. Complete the FAR pipeline once so Gold tables exist and are populated.
2. Open **`<NOTEBOOK_PREFIX>_05_Agent`**, run its SDK install cell, allow any requested kernel restart, then run the deploy cell.

Verify source availability, publication and evaluation results before use. Keep this Agent restricted to central reviewers; owner-report RLS does not secure it.

## 6. Optional: FUAM-targeted reviews and native Outlook owner email

Both features are optional; the normal FAR pipeline remains available without them.

After a successful scoped review, open **`<NOTEBOOK_PREFIX>_06_TargetedReviewSetup`**. Set `DEPLOY_TARGETED_REVIEW="true"`, configure the FUAM source and review the ranking parameters. Keep the FAR output Lakehouse attached and leave stamped advanced wiring unchanged.

06 creates a **parent pipeline**, **Runner** and **Completion** notebook. Test the parent with an explicit `WORKSPACE_IDS` candidate allow-list and notifications off. Empty selection skips FAR; source failures stop it. The parent does not inherit standalone FAR's saved workspace scope.

For email, follow the [deployment walkthrough](https://github.com/microsoft/fabric-architecture-review/blob/main/fabric/DEPLOYMENT.md): connect and activate the parent's **Office 365 Outlook** activity, set `FAR_REPORT_URL`, then test with `NOTIFICATIONS_ENABLED="true"`. Setup sends nothing. Messages go to eligible direct-user workspace administrators, not automatically to the sender.

Email provides review context and a link, not a task list or access grant. Use the separately validated owner report for workspace owners; a filtered governance link is not an access boundary. Confirm delivery in native monitoring and the mailbox before scheduling. Inspect uncertain outcomes before retrying to avoid duplicate mail.

Schedule the validated parent for recurring targeted reviews, and keep the independent daily 07 access sync when owner reporting is enabled. Setup enables neither schedule.

On redeployment, pause schedules and finish active runs before rerunning 06. It preserves configured email settings but resets the notification default to false and restores generated message bindings. Check schedule overrides before resuming. See the [targeted-review reference](https://github.com/microsoft/fabric-architecture-review/blob/main/docs/targeted-review.md) for parameters and recovery.
